# Reproducing ResNet on CIFAR-10 (Kaggle GPU)

Kaggle port of `notebooks/reproduce_colab.ipynb`. Same repo, same driver, same schedule.

**Settings > Accelerator > GPU T4 x2 (or P100) before running.**

Run this with **Save Version > Save & Run All (Commit)**: it executes headless, so no browser
tab and no awake laptop are needed, and `/kaggle/working/results.csv` is persisted with the
version. Seeds already present in the repo's committed `results.csv` are skipped.

In [ ]:
!git clone -q https://github.com/AmroAbujabal/resnet-cifar-repro.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install pyyaml pytest
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Results live in /kaggle/working (persisted with the saved version), seeded from the repo's
# committed results.csv so the seeds already reported are carried forward and skipped.
import csv, os, shutil
RESULTS = '/kaggle/working/results.csv'
if not os.path.exists(RESULTS):
    shutil.copy('results.csv', RESULTS)

def done(model, seed):
    with open(RESULTS) as f:
        return any(r['model'] == model and int(r['seed']) == seed for r in csv.DictReader(f))

print(open(RESULTS).read())

In [ ]:
# Full suite, including the T1 data tests that download CIFAR-10 (and CIFAR-100
# for the two Phase 3 tests) -- the Toronto mirror is slow, allow ~15 min.
!python -m pytest -q


In [ ]:
# ResNet-20, seeds 0-2 (paper Table 6: 8.75%). ~51 min/seed on a T4.
for s in (0, 1, 2):
    if done('resnet20', s):
        print(f'skip resnet20 seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/resnet20.yaml --seed {s} --device cuda --results {RESULTS}

In [ ]:
# ResNet-56, seeds 0-2 (paper Table 6: 6.97%). ~2.5 h/seed on a T4 -- check this fits the
# session limit before committing; if not, run seeds one at a time across versions.
for s in (0, 1, 2):
    if done('resnet56', s):
        print(f'skip resnet56 seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/resnet56.yaml --seed {s} --device cuda --results {RESULTS}

## Phase 3 — pre-activation × CIFAR-100 (2×2)

Four cells: {original, pre-act} ResNet-56 × {CIFAR-10, CIFAR-100}. **original × CIFAR-10 is
already done** (3 seeds in `results.csv`), so three remain, ~6.3 h each (3 seeds × ~2.1 h).

Kaggle kills a session at its limit and **discards `/kaggle/working`**, so run **one config per
saved version**: set `PHASE3` below, Save & Run All, then pull the output, commit `results.csv`,
push, and come back for the next one.


In [ ]:
# Set this per version -- do NOT run more than one config in a single version.
PHASE3 = 'preact56'          # then 'resnet56_c100', then 'preact56_c100'

for s in (0, 1, 2):
    if done(PHASE3, s):
        print(f'skip {PHASE3} seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/{PHASE3}.yaml --seed {s} --device cuda --results {RESULTS}


In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS)
display(df)
print(df.groupby('model')[['test_error_pct', 'train_error_pct']].agg(['mean', 'std', 'count']))
